In [2]:
import json
import os
import classify_paper
import pandas as pd
from typing import Tuple

PRINT_RANK="A*"

crossref_mapping = {}

def get_core_rank(pub: dict) -> Tuple[str, bool, object]:
    year = int(pub.get("year", 0))
    # Extract venue key from paper key or crossref
    venue_crossref = pub.get("crossref", "")
    venue_dblp = ""
    if venue_crossref:
        venue_dblp = '/'.join(venue_crossref.split('/')[:2])
    else:
        key = pub.get("key", "")
        if key:
            venue_dblp = '/'.join(key.split('/')[:2])

    venue_name = pub.get("venue", "")

    global crossref_mapping
    if venue_dblp not in crossref_mapping:
        crossref_mapping[venue_dblp] = []
    crossref_mapping[venue_dblp].append(pub)

    # Determine rank using the normalized dblp/acronym lookup
    rank, short_paper, acronym_row = classify_paper.core_rank(venue_name, venue_crossref, venue_dblp, year)

    # If the core table provides a fullpaper threshold for this venue, use it
    threshold = None
    if acronym_row is not None:
        try:
            if hasattr(acronym_row, 'get'):
                thr = acronym_row.get('fullpaper_threshold')
            else:
                thr = None
            if thr is not None and str(thr).strip() != "":
                threshold = int(str(thr).strip())
        except Exception:
            threshold = None

    if not short_paper and threshold is not None:
        pages_str = pub.get("pages", "")
        pagenum = classify_paper.get_paper_length(pages_str)
        if pagenum is None:
            print(f"No page number for {pub}")
        elif pagenum < threshold:
            short_paper = True

    return rank, short_paper, acronym_row

with open(os.path.join('..', 'proceedings.json'), 'r', encoding='utf-8') as f:
    data = json.load(f)

core_table = pd.read_csv(os.path.join("../inputs", "core_table.csv"))

rank_number = {}
found_venues = {}
found_crossrefs = {}
crossref_to_name = {}

for key, venues in data.items():
    for year, venue in venues.items():
        crossrefs=venue.get("key").split('/')
        crossref_short=f"{crossrefs[0]}/{crossrefs[1]}"
        pub = {
            "type": venue.get("type"),
            "authors": [],
            "title": venue.get("title"),
            "year": venue.get("year"),
            "url": "",
            "venue": venue.get("venue"),
            "crossref": crossref_short,
            "pages": "1-12"
        }
        if pub.get("venue") is None:
            #print(f"Missing venue for {pub}")
            continue

        found_crossrefs[crossref_short]=1
        desc = f"{pub.get("venue", "")} {pub.get("title", "")}"
        if crossref_short not in crossref_to_name:
            crossref_to_name[crossref_short] = desc
        else:
            same=False
            for publ in crossref_to_name.get(crossref_short, "").split('\n'):
                if publ == desc:
                    same=True
            if not same:
                crossref_to_name[crossref_short] +='\n' + desc
        rank, short_paper, acronym_row = get_core_rank(pub)
        #print(f"{pub['venue']} ({pub['year']}): {rank} - {short_paper}")
        rank_str = f"{rank} - {'short' if short_paper else 'full'} {venue.get('type')}"
        rank_number[rank_str] = rank_number.get(rank_str, 0) + 1
        if acronym_row is not None:
            id = acronym_row.get("ID")
            if acronym_row is not None and id is not None:
                id_str=str(id).upper()
                if id_str not in found_venues:
                    found_venues[id_str] = []
                found_venues[id_str].append(pub)

for rank_str, count in rank_number.items():
    print(f"{rank_str}: {count}")



# Check core_table for missing IDs and count by computed rank
no_dblps = 0
missing_counts = {}
found_counts = {}
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    dblp_venues=row.get("dblp_venue")
    # Treat NaN as empty: require non-NaN and non-blank string
    if pd.notna(dblp_venues) and str(dblp_venues).strip() != "":
        for dblp_venue in str(dblp_venues).split(';'):
            if dblp_venue not in found_crossrefs:
                found_crossrefs[dblp_venue]=0
            else:
                found_crossrefs[dblp_venue]=2
    else:
        no_dblps+=1
    best_rank= "no_rank"
    core_ranks_str = row.get("YearsListed", "")
    if "_A*" in core_ranks_str:
        best_rank = "A*"
    elif '_A' in core_ranks_str:
        best_rank = "A"
    elif '_B' in core_ranks_str:
        best_rank = "B"
    elif '_C' in core_ranks_str:
        best_rank = "C"
            
    cid = str(row.get("ID")).upper()
    if cid not in found_venues:
        if core_ranks_str!="ERA2010_A":
            missing_counts[best_rank] = missing_counts.get(best_rank, 0) + 1
            if best_rank == PRINT_RANK:
                if row['dblp_venue'] is not None and row['dblp_venue'] != "":
                    print(f"Missing {best_rank} venue: {row['dblp_venue']} {row.get('Acronym', '')} {row.get('Name', '')} {row.get('YearsListed')} (ID: {cid})")
                    venue_crossref = row.get("dblp_venue", "")
                    if venue_crossref in crossref_mapping:
                        print(f"Found in crossref_mapping: {venue_crossref} {row.get('Acronym', '')} {row.get('Name', '')}")
                        for pub in crossref_mapping[venue_crossref]:
                            print(f">{pub.get('venue')}< : {pub["title"]} ({pub['year']}) - {pub.get('type')})")
    else:
        found_counts[best_rank] = found_counts.get(best_rank, 0) + 1

print("\nMissing counts by rank category:")
for k, v in missing_counts.items():
    print(f"{k}: {v}")

print("\nFound counts by rank category:")
for k, v in found_counts.items():
    print(f"{k}: {v}")

not_in_proceedings=[crossref for crossref, found in found_crossrefs.items() if found==0]
not_in_core_table=[crossref for crossref, found in found_crossrefs.items() if found==1 and crossref.startswith('conf/')]
print(f"Crossrefs not found in proceeding.json: {len(not_in_proceedings)}")
print(f"Crossrefs not found in core_table_csv: {len(not_in_core_table)}")
print(f"Core table rows with no dblp venue: {no_dblps}")

print(f"\nNot in proceedings.json: {not_in_proceedings}")
print(f"\nNot in core_table_csv: {not_in_core_table}")

no_rank - full proceedings: 29520
B - short proceedings: 3251
C - short proceedings: 2042
A - full proceedings: 5156
A - short proceedings: 3836
C - full proceedings: 5713
B - full proceedings: 7658
A* - full proceedings: 2228
A* - short proceedings: 2675
no_rank - short proceedings: 393
TBR - full proceedings: 2

Missing core_table entries:

Missing counts by rank category:
A: 14
B: 177
C: 516
no_rank: 18

Found counts by rank category:
A*: 84
A: 313
B: 434
C: 517
no_rank: 47
Crossrefs not found in proceeding.json: 10
Crossrefs not found in core_table_csv: 5158
Core table rows with no dblp venue: 866

Not in proceedings.json: ['journals/tog', 'journals/cgf', 'conf/ocis', 'conf/openarch', 'series/eptcs', 'conf/ijcrs', 'conf/tdb', 'conf/jcis', 'conf/krdb', 'conf/pdis']

Not in core_table_csv: ['conf/psl', 'conf/africom', 'conf/clrimg', 'conf/acl-dmtw', 'conf/ate', 'conf/e2dc', 'conf/secitc', 'conf/ised', 'conf/termgraph', 'conf/vision', 'conf/sigcommops', 'conf/infoseccd', 'conf/acleven

In [3]:
import os
import json

counts_path = os.path.join('..', 'dblp_venue_counts.json')
try:
    with open(counts_path, 'r', encoding='utf-8') as f:
        dblp_counts = json.load(f)
except Exception as e:
    print(f"Nem sikerült betölteni {counts_path}: {e}")
    dblp_counts = {}

# A korábbi cellában létrehozott változó: not_in_core_table
not_in = globals().get('not_in_core_table')
if not_in is None:
    print("Figyelmeztetés: 'not_in_core_table' változó nem található a kernelben. Üres listát használok.")
    not_in = []

# Párosítás (crossref_short -> count) és rendezés
rows = [(v, int(dblp_counts.get(v, 0))) for v in not_in]
rows_sorted = sorted(rows, key=lambda x: x[1], reverse=True)

# Statisztikák
total_missing = len(rows)
with_count = sum(1 for _, c in rows if c > 0)
sum_counts = sum(c for _, c in rows)

print(f"Összes 'not_in_core_table' elem: {total_missing}")
print(f"Ebből DBLP-ben talált (count>0): {with_count}")
print(f"DBLP-ben összes előfordulás összesen: {sum_counts}")

# Top 50
top50 = rows_sorted[:50]
print('\nTop 50 leggyakoribb not_in_core_table elemek (crossref_short: count):')
for v, c in top50:
    print(f"{v}: {c} \n{crossref_to_name.get(v, 'N/A')}")

# Mentsük el az eredményt a results mappába
os.makedirs('results', exist_ok=True)
out_path = os.path.join('results', 'not_in_core_table_sorted_by_dblp_counts.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump({'stats': {'total': total_missing, 'with_count': with_count, 'sum_counts': sum_counts}, 'top50': top50, 'all': rows_sorted}, f, ensure_ascii=False, indent=2)

print(f"\nRendezett lista és statisztika elmentve: {out_path}")

Összes 'not_in_core_table' elem: 5158
Ebből DBLP-ben talált (count>0): 5146
DBLP-ben összes előfordulás összesen: 1299459

Top 50 leggyakoribb not_in_core_table elemek (crossref_short: count):
conf/amcc: 24284 
ACC American Control Conference, ACC 2010, Baltimore, Maryland, USA, June 30 - July 2, 2010
ACC Proceedings of the 2004 American Control Conference, ACC 2004, Boston, MA, USA, June 30 - July 2, 2004
ACC American Control Conference, ACC 2024, Toronto, ON, Canada, July 10-12, 2024
ACC American Control Conference, ACC 2022, Atlanta, GA, USA, June 8-10, 2022
ACC 2021 American Control Conference, ACC 2021, New Orleans, LA, USA, May 25-28, 2021
ACC American Control Conference, ACC 2013, Washington, DC, USA, June 17-19, 2013
ACC 2025 American Control Conference, ACC 2025, Denver, CO, USA, July 8-10, 2025
ACC American Control Conference, ACC 2002, Anchorage, Alaska, USA, May 8-10 2002.
ACC American Control Conference, ACC 2023, San Diego, CA, USA, May 31 - June 2, 2023
ACC American Cont

In [4]:
import os
import json
import re
import pandas as pd
import difflib
import requests

# Load proceedings and core table
with open(os.path.join('..', 'proceedings.json'), 'r', encoding='utf-8') as f:
    data = json.load(f)
core_table = pd.read_csv(os.path.join('../inputs', 'core_table.csv'))

# Stopwords for token comparison
stopwords = set([
    'and','of','the','in','on','for','with','international','conference',
    'symposium','proceedings','journal','transactions','workshop','letters',
    'systems','system','applications','application','computing','computers',
    'review','annals','bulletin','ieee','acm','springer','elsevier','was',
    'joint','congress','society','education','association',
])

# Stricter thresholds
JACCARD_THRESH = 0.15
FRAC_TITLE_THRESH = 0.25
FUZZY_THRESH = 0.85
MAX_FOUND = 100
print(f"Using stricter thresholds: jaccard={JACCARD_THRESH}, frac_title={FRAC_TITLE_THRESH}, fuzzy={FUZZY_THRESH}")

# Use LLM if OPENAI_API_KEY is set
USE_LLM = bool(os.getenv('OPENAI_API_KEY'))
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')

# Embedding model settings
EMB_MODEL = os.getenv('EMB_MODEL', 'all-MiniLM-L6-v2')
EMB_AVAILABLE = False
_emb_model = None
_core_embs = None

found_venues = set()
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    if row['dblp_venue']:
        found_venues.add(row['dblp_venue'])

def tokenize(text):
    parts = re.split(r"[^\wÀ-ž]+", (text or ''))
    toks = set()
    for p in parts:
        p = p.strip().lower()
        if not p:
            continue
        if len(p) <= 2:
            continue
        if p in stopwords:
            continue
        toks.add(p)
    return toks


def fuzzy_similar_count(set_a, set_b, thresh=FUZZY_THRESH, max_tokens=8):
    """Return number of fuzzy token pairs with similarity >= thresh.

    Compare up to `max_tokens` tokens from each set to keep runtime bounded.
    """
    if not set_a or not set_b:
        return 0
    a_list = sorted(list(set_a))[:max_tokens]
    b_list = sorted(list(set_b))[:max_tokens]
    count = 0
    for a in a_list:
        for b in b_list:
            if difflib.SequenceMatcher(None, a, b).ratio() >= thresh:
                count += 1
                if count >= 5:
                    return count
    return count


def ask_llm(item, api_key=None, model=None, timeout=20):
    api_key = api_key or os.getenv('OPENAI_API_KEY')
    if not api_key:
        return {'error': 'no_api_key'}
    model = model or LLM_MODEL

    proceedings = {
        'key': item.get('proceedings_key'),
        'year': item.get('year'),
        'title': item.get('title'),
        'venue_value': item.get('venue_value'),
        'crossref_short': item.get('crossref_short'),
    }
    core = {
        'core_name': item.get('core_name'),
        'core_acronym': item.get('core_acronym'),
        'core_dblp_venue': item.get('core_dblp_venue'),
        'common_tokens': item.get('common_tokens'),
        'jaccard': item.get('jaccard'),
        'common_count': item.get('common_count')
    }

    prompt = (
        "Decide whether the following Proceedings entry and Core table entry refer to the same conference or series. "
        "Answer in JSON format exactly: {\"same\": \"yes|no|maybe\", \"score\": <0.0-1.0>, \"reason\": \"...\"}.\n\n"
        f"Proceedings: {json.dumps(proceedings, ensure_ascii=False)}\n\n"
        f"Core: {json.dumps(core, ensure_ascii=False)}\n\n"
        "Consider title tokens, acronyms, dblp crossref, and similarity metrics. Be concise."
    )

    url = 'https://api.openai.com/v1/chat/completions'
    headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant that classifies whether two venue records refer to the same conference.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 200,
        'temperature': 0.0,
    }

    try:
        r = requests.post(url, headers=headers, json=payload, timeout=timeout)
        r.raise_for_status()
        j = r.json()
        text = j['choices'][0]['message']['content']
        try:
            out = json.loads(text)
            return out
        except Exception:
            import re as _re
            m = _re.search(r"\{.*\}", text, _re.DOTALL)
            if m:
                try:
                    return json.loads(m.group(0))
                except Exception:
                    return {'error': 'parse_failed', 'raw': text}
            return {'error': 'no_json', 'raw': text}
    except Exception as e:
        return {'error': 'request_failed', 'detail': str(e)}


# LLM comparison cell (prints concise JSON samples)
def compare_with_llm(suspicious_list=None, max_show=20):
    lst = suspicious_list if suspicious_list is not None else globals().get('suspicious', [])
    cnt = len(lst)
    print(f'compare_with_llm: {cnt} items', flush=True)
    if cnt == 0:
        return
    to_show = min(max_show, cnt)
    for i, s in enumerate(lst[:to_show]):
        print('---', flush=True)
        print(f'[{i+1}/{cnt}] Proceedings: {s.get("proceedings_key", "")} / {s.get("year", "")}', flush=True)
        try:
            print(json.dumps(s, ensure_ascii=False, indent=2), flush=True)
        except Exception:
            print(repr(s), flush=True)

        if USE_LLM:
            res = ask_llm(s)
            try:
                print('LLM verdict:', json.dumps(res, ensure_ascii=False, indent=2), flush=True)
            except Exception:
                print('LLM verdict:', repr(res), flush=True)
        elif EMB_AVAILABLE:
            res = ask_llm_embedding(s)
            try:
                print('Embedding verdict:', json.dumps(res, ensure_ascii=False, indent=2), flush=True)
            except Exception:
                print('Embedding verdict:', repr(res), flush=True)
        else:
            print('LLM/Embedding not enabled (set OPENAI_API_KEY or install sentence-transformers)', flush=True)

    if cnt > to_show:
        print(f'... and {cnt - to_show} more', flush=True)


# Precompute token sets for core_table rows
core_rows = []
for i, row in core_table.iterrows():
    name = row.get('Name', '')
    skip=False
    for dblp_venue in str(row.get('dblp_venue', '')).split(';'):
        if dblp_venue.strip() in found_venues:
            skip=True
    if skip:
        continue 
    toks = tokenize(name)
    core_rows.append({
        'index': i,
        'Name': name,
        'Acronym': str(row.get('Acronym', '')).strip(),
        'dblp_venue': str(row.get('dblp_venue', '')).strip(),
        'tokens': toks,
        'row': row
    })

# Try to load embedding model and precompute core embeddings
try:
    from sentence_transformers import SentenceTransformer, util
    _emb_model = SentenceTransformer(EMB_MODEL)
    core_texts = [cr['Name'] or '' for cr in core_rows]
    if core_texts:
        _core_embs = _emb_model.encode(core_texts, convert_to_tensor=True)
    EMB_AVAILABLE = True
    print(f'Embedding model loaded ({EMB_MODEL}), precomputed {len(core_texts)} core embeddings', flush=True)
except Exception as e:
    EMB_AVAILABLE = False
    _emb_model = None
    _core_embs = None
    print('Embedding model not available:', e, flush=True)


def ask_llm_embedding(item, top_k=1):
    if not EMB_AVAILABLE or _emb_model is None or _core_embs is None:
        return {'error': 'embeddings_unavailable'}
    proc_text = ' '.join(filter(None, [str(item.get('title', '')), str(item.get('venue_value', '')), str(item.get('crossref_short', ''))]))
    proc_emb = _emb_model.encode(proc_text, convert_to_tensor=True)
    scores = util.cos_sim(proc_emb, _core_embs)[0]
    best_idx = int(scores.argmax())
    best_score = float(scores[best_idx])
    best_core = core_rows[best_idx]
    if best_score >= 0.72:
        same = 'yes'
    elif best_score >= 0.55:
        same = 'maybe'
    else:
        same = 'no'
    reason = f"embedding_score={best_score:.3f}; core_name='{best_core['Name']}'"
    return {'same': same, 'score': best_score, 'reason': reason, 'best_core_index': best_idx}


# Find suspicious matches with stricter heuristic
suspicious = []
found = 0
stop = False
for key, venues in data.items():
    if stop:
        break
    for year, v in venues.items():
        title = v.get('title', '')
        venue_val = v.get('venue', '')
        crossref = v.get('key', '') or v.get('crossref', '')
        crossref_short = '/'.join(str(crossref).split('/')[:2]) if crossref else ''
        if crossref_short in found_venues:
            continue 
        title_toks = tokenize(title)
        if not title_toks:
            continue
        for cr in core_rows:
            name_toks = cr['tokens']
            if not name_toks:
                continue
            common = title_toks & name_toks
            union = title_toks | name_toks
            common_count = len(common)
            jaccard = (len(common) / len(union)) if union else 0.0
            frac_common_title = (len(common) / len(title_toks)) if title_toks else 0.0

            # Compute fuzzy matches as bounded fallback
            fuzzy_matches = 0
            if jaccard < JACCARD_THRESH and len(title_toks) * len(name_toks) <= 200:
                fuzzy_matches = fuzzy_similar_count(title_toks, name_toks)

            # Stricter rule: require >=2 shared tokens AND at least one of jaccard/frac thresholds,
            # or require stronger fuzzy evidence (>=3 fuzzy token matches)
            significant = False
            if (common_count >= 3) and (jaccard >= JACCARD_THRESH or frac_common_title >= FRAC_TITLE_THRESH):
                significant = True
            elif fuzzy_matches >= 3:
                significant = True

            if significant:
                # Normalize values
                venue_val_l = str(venue_val or '').strip().lower()
                acro = str(cr['Acronym'] or '').strip().lower()
                dblp = str(cr['dblp_venue'] or '').strip()
                crossref_short_s = str(crossref_short or '').strip()
                # extract second tag from crossref (if present) and compare lowercased to acronym
                crossref_second = ''
                if '/' in crossref_short_s:
                    parts = crossref_short_s.split('/')
                    if len(parts) > 1:
                        crossref_second = parts[1].lower()
                # acro_match if acronym matches venue value or matches crossref second tag (case-insensitive)
                acro_match = (acro != '' and (acro == venue_val_l or (crossref_second != '' and acro == crossref_second)))
                dblp_match = (dblp == crossref_short_s)
                # only collect when NEITHER acronym nor dblp match
                if (not acro_match) and (not dblp_match):
                    suspicious.append({
                        'proceedings_key': key,
                        'year': year,
                        'title': title,
                        'venue_value': venue_val,
                        'crossref_short': crossref_short,
                        'core_index': cr['index'],
                        'core_name': cr['Name'],
                        'core_acronym': cr['Acronym'],
                        'core_dblp_venue': cr['dblp_venue'],
                        'common_tokens': sorted(list(common)),
                        'common_count': common_count,
                        'jaccard': round(jaccard, 3),
                        'frac_common_title': round(frac_common_title, 3),
                        'acro_match': acro_match,
                        'dblp_match': dblp_match
                    })
                    found += 1
                    if found >= MAX_FOUND:
                        stop = True
                        break
        if stop:
            break

print(f'Found {found} suspicious pairs to review (limited to {MAX_FOUND})', flush=True)

try:
    compare_with_llm(suspicious, max_show=MAX_FOUND)
except Exception as e:
    print(f'compare_with_llm not available or failed: {e}', flush=True)


Using stricter thresholds: jaccard=0.15, frac_title=0.25, fuzzy=0.85

Missing core_table entries:


/Users/janostapolcai/Desktop/janos/fejlesztes/coreA/corePaperList/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7901.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded (all-MiniLM-L6-v2), precomputed 880 core embeddings
Found 100 suspicious pairs to review (limited to 100)
compare_with_llm: 100 items
---
[1/100] Proceedings: series/isrl / 97
{
  "proceedings_key": "series/isrl",
  "year": "97",
  "title": "Intelligent Decision Making in Quality Management - Theory and Applications",
  "venue_value": null,
  "crossref_short": "series/isrl",
  "core_index": 1010,
  "core_name": "Australian Conference for Knowledge Management and Intelligent Decision Support",
  "core_acronym": "ACKMIDS",
  "core_dblp_venue": "nan",
  "common_tokens": [
    "decision",
    "intelligent",
    "management"
  ],
  "common_count": 3,
  "jaccard": 0.333,
  "frac_common_title": 0.5,
  "acro_match": false,
  "dblp_match": false
}
Embedding verdict: {
  "same": "no",
  "score": 0.5184003114700317,
  "reason": "embedding_score=0.518; core_name='Australian Conference for Knowledge Management and Intelligent Decision Support'",
  "best_core_index": 243
}
---

In [5]:
MAX_FOUND = 100
# Find suspicious matches with stricter heuristic
acronyms = {}
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    dblp = row.get('dblp_venue')
    # consider rows missing dblp_venue (NaN or blank)
    if pd.notna(dblp) and str(dblp).strip() != '':
        continue
    acr_col = row.get('Acronym')
    if pd.isna(acr_col) or str(acr_col).strip() == '':
        continue
    for acronym in str(acr_col).split(';'):
        a = acronym.strip().lower()
        if a:
            acronyms[a] = {'Name': row.get('Name'), 'Acronym': row.get('Acronym'), 'YearsListed': row.get('YearsListed')}

suspicious = []
stop = False
printed_crossref=[]
for key, venues in data.items():
    if stop:
        break
    for year, v in venues.items():
        title = v.get('title', '')
        venue_val = v.get('venue', '') or ''
        venue_norm = str(venue_val).strip().lower()
        crossref = v.get('key', '') or v.get('crossref', '')
        crossref_short = '/'.join(str(crossref).split('/')[:2]) if crossref else ''
        if crossref_short in found_venues:
            continue
        if crossref_short in printed_crossref:
            continue
        if venue_norm and venue_norm in acronyms:
            cr = acronyms[venue_norm]
            suspicious.append({
                'proceedings_key': key,
                'year': year,
                'title': title,
                'venue_value': venue_val,
                'crossref_short': crossref_short,
                'core_name': cr.get('Name'),
                'core_acronym': cr.get('Acronym'),
                'core_yearslisted': cr.get('YearsListed')
            })
            printed_crossref.append(crossref_short)
            if len(suspicious) >= MAX_FOUND:
                stop = True
                break

print(f'Found {len(suspicious)} suspicious pairs to review (limited to {MAX_FOUND})', flush=True)

try:
    compare_with_llm(suspicious, max_show=MAX_FOUND)
except Exception as e:
    print(f'compare_with_llm not available or failed: {e}', flush=True)



Missing core_table entries:
Found 23 suspicious pairs to review (limited to 100)
compare_with_llm: 23 items
---
[1/23] Proceedings: conf/gi-ws / 2023
{
  "proceedings_key": "conf/gi-ws",
  "year": "2023",
  "title": "IEEE/ACM International Workshop on Genetic Improvement, GI@ICSE 2023, Melbourne, Australia, May 20, 2023",
  "venue_value": "GI",
  "crossref_short": "conf/gi-ws",
  "core_name": "IEEE Global Internet Symposium",
  "core_acronym": "GI",
  "core_yearslisted": "ERA2010_C, CORE2013_C, CORE2014_C, CORE2017_C, CORE2018_C, CORE2020_C, CORE2021_C, CORE2023_C, ICORE2026_C"
}
Embedding verdict: {
  "same": "no",
  "score": 0.42467033863067627,
  "reason": "embedding_score=0.425; core_name='The Annual International Conference of the IEEE Engineering in Medicine and Biology Society'",
  "best_core_index": 823
}
---
[2/23] Proceedings: conf/ais2 / 2010
{
  "proceedings_key": "conf/ais2",
  "year": "2010",
  "title": "Autonomous and Intelligent Systems - First International Conference

In [6]:
# Listázza azokat az eseteket, ahol a core `Acronym` megegyezik a crossref második tagjával (hatékony változat)
import os
import json
from collections import defaultdict

# Build found_venues set from core_table dblp_venue to skip already-known venues
found_venues = set()
print("\nMissing core_table entries:")
for _, row in core_table.iterrows():
    if row['dblp_venue']:
        found_venues.add(row['dblp_venue'])
        

# Build acronym -> core rows map (lowercased) from precomputed core_rows for O(1) lookup
acro_map = defaultdict(list)
for i, cr in core_table.iterrows():
    a = str(cr.get('Acronym','')).strip().lower()
    if a:
        acro_map[a].append({'index': i, 'Acronym': cr.get('Acronym'), 'Name': cr.get('Name'), 'YearsListed': cr.get('YearsListed'), 'dblp_venue': cr.get('dblp_venue')})

# Flatten proceedings entries to a list for stable iteration and progress reporting
entries = [(k, y, v) for k, vs in data.items() for y, v in vs.items()]

# Progress wrapper: prefer console tqdm, fallback to notebook tqdm, else identity
try:
    from tqdm import tqdm
    progress = lambda it, **kw: tqdm(it, **kw)
except Exception:
    try:
        from tqdm.notebook import tqdm
        progress = lambda it, **kw: tqdm(it, **kw)
    except Exception:
        progress = lambda it, **kw: it

matches = []
for key, year, v in progress(entries, desc="Checking crossref-second", unit="entry"):
    crossref = v.get('key') or ''
    if not crossref:
        continue
    parts = str(crossref).split('/')
    if len(parts) < 2:
        continue
    if parts[0] != 'conf':
        continue
    short = f"{parts[0]}/{parts[1]}"
    if short in found_venues:
        continue
    second = parts[1].strip().lower()
    if second in acro_map:
        for core_row in acro_map[second]:
            matches.append({
                'proceedings_key': key,
                'year': year,
                'title': v.get('title'),
                'venue_value': v.get('venue'),
                'crossref_short': short,
                'crossref_second': second,
                'core_index': core_row['index'],
                'core_acronym': core_row.get('Acronym'),
                'core_name': core_row.get('Name'),
                'core_rank': core_row.get('YearsListed')
            })

print(f'Found {len(matches)} acronym == crossref-second matches')
# write results to results/acro_crossref_matches.json
os.makedirs('results', exist_ok=True)
out_path = os.path.join('results', 'acro_crossref_matches.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(matches, f, ensure_ascii=False, indent=2)
print(f'Wrote {len(matches)} matches to {out_path}', flush=True)


Missing core_table entries:


Checking crossref-second: 100%|██████████| 63000/63000 [00:00<00:00, 887349.52entry/s]

Found 774 acronym == crossref-second matches
Wrote 774 matches to results/acro_crossref_matches.json


In [7]:
# Scan core_table for duplicate non-empty `Acronym` values and print matches; also report empty Acronym rows
import pandas as pd
import numpy as np
# Normalize acronyms (lowercase, stripped)
acro_norm = core_table['Acronym'].fillna('').astype(str).str.strip().str.lower()
core_table['_acro_norm'] = acro_norm
# Consider only non-empty acronyms for duplicate detection
non_empty_mask = core_table['_acro_norm'] != ''
counts = core_table.loc[non_empty_mask, '_acro_norm'].value_counts()
dups = counts[counts > 1]
if dups.empty:
    print('No duplicated non-empty Acronym values found')
else:
    print(f"Found {len(dups)} duplicated non-empty acronym(s): {', '.join(dups.index.tolist())}")
    for ac in dups.index:
        print('\n=== Acronym:', ac, '===')
        display_cols = ['ID','Acronym','Name','YearsListed','dblp_venue']
        cols = [c for c in display_cols if c in core_table.columns]
        print(core_table[core_table['_acro_norm'] == ac][cols].to_string(index=False))
# Report rows with empty acronym separately
empty_rows = core_table[core_table['_acro_norm'] == '']
if empty_rows.shape[0] > 0:
    print(f"\nRows with empty Acronym: {empty_rows.shape[0]} (showing up to 20 rows)")
    display_cols = ['ID','Acronym','Name','YearsListed','dblp_venue']
    cols = [c for c in display_cols if c in core_table.columns]
    print(empty_rows[cols].head(20).to_string(index=False))
# cleanup temporary column
core_table.drop(columns=['_acro_norm'], inplace=True, errors='ignore')

Found 52 duplicated non-empty acronym(s): isc, icis, icsc, iccc, cisis, sac, cec, wcc, sigchi-nz, ais, ai, ism, dac, ipc, isarc, icmr, acsac, iccs, ecrypt, itc, dsn, iswc, trb, cis, icci, grc, gi, icics, sc, icannga, sea, embc, apcom, iccsa, xp, hai, icvs, ismm, inap, icec, ices, icspc, iccp, ie, edm, acac, dicta, osdi, caia, cacsd, cca, ias

=== Acronym: isc ===
  ID Acronym                                                               Name                                                                                                            YearsListed         dblp_venue
 827     ISC                                    Information Security Conference CORE2008_C, ERA2010_C, CORE2013_C, CORE2014_C, CORE2017_B, CORE2018_B, CORE2020_B, CORE2021_B, CORE2023_C, ICORE2026_C                NaN
 820     ISC                                   Industrial Simulation Conference             ERA2010_C, CORE2013_C, CORE2014_C, CORE2017_C, CORE2018_C, CORE2020_C, CORE2021_C, CORE2023_C, ICORE2026_C

In [8]:
# Update `core_table.csv` from `results/acro_crossref_matches.json` for unique acronyms (only fill empty `dblp_venue`)
import os
import json
import pandas as pd
# Paths
matches_path = os.path.join('results','acro_crossref_matches_.json')
core_in_path = os.path.join('..','inputs','core_table.csv')
core_out_path = os.path.join('..','inputs','core_table_.csv')
if not os.path.exists(matches_path):
    print(f'Matches file not found: {matches_path}')
else:
    with open(matches_path,'r', encoding='utf-8') as f:
        matches = json.load(f)
    # Build map: normalized acronym -> set of crossref_short values
    acro_to_cross = {}
    for m in matches:
        ac = str(m.get('core_acronym') or '').strip().lower()
        cross = str(m.get('crossref_short') or '').strip()
        if not ac:
            continue
        acro_to_cross.setdefault(ac, set()).add(cross)
    # Load core table
    if not os.path.exists(core_in_path):
        print(f'Core table not found: {core_in_path}')
    else:
        df = pd.read_csv(core_in_path)
        df['_acro_norm'] = df['Acronym'].fillna('').astype(str).str.strip().str.lower()
        # find duplicated non-empty acronyms
        dup_counts = df.loc[df['_acro_norm']!='', '_acro_norm'].value_counts()
        dups = set(dup_counts[dup_counts>1].index.tolist())
        changes = []
        for ac, cross_set in sorted(acro_to_cross.items()):
            if ac in dups:
                print(f"Skipping {ac!r}: acronym is duplicated in core_table")
                continue
            # filter out empty crossref values
            nonempty = [c for c in sorted(cross_set) if c]
            if not nonempty:
                print(f"Skipping {ac!r}: no crossref_short available in matches")
                continue
            uniq = sorted(set(nonempty))
            if len(uniq) != 1:
                print(f"Skipping {ac!r}: multiple crossref_short values found: {uniq}")
                continue
            value = uniq[0]
            rows = df.index[df['_acro_norm'] == ac].tolist()
            if not rows:
                print(f"No core_table row with Acronym {ac!r} found")
                continue
            if len(rows) > 1:
                print(f"Skipping {ac!r}: unexpected multiple core rows {rows}")
                continue
            idx = rows[0]
            # only fill when existing dblp_venue is empty or NaN
            old_val = df.at[idx, 'dblp_venue'] if 'dblp_venue' in df.columns else None
            is_empty = pd.isna(old_val) or str(old_val).strip() == ''
            if is_empty:
                df.at[idx, 'dblp_venue'] = value
                changes.append((ac, int(df.at[idx, 'ID']) if 'ID' in df.columns else idx, old_val, value))
                print(f"Filled Acronym {ac!r} at row {idx}: dblp_venue {old_val!r} -> {value!r}")
            else:
                if f"conf/{ac.lower()}"!=old_val.lower(): 
                    print(f"Skipping update for {ac!r}: existing non-empty dblp_venue kept ({old_val!r})")
        # Write updated core table copy
        df.drop(columns=['_acro_norm'], inplace=True, errors='ignore')
        df.to_csv(core_out_path, index=False)
        print(f"Wrote updated core table to {core_out_path} ({len(changes)} changes)")
        if changes:
            print('Sample changes:')
            for c in changes[:20]:
                print(c)

Skipping update for 'acai': existing non-empty dblp_venue kept ('conf/acai3')
Skipping 'ai': acronym is duplicated in core_table
Skipping 'ais': acronym is duplicated in core_table
Skipping update for 'arcs': existing non-empty dblp_venue kept ('conf/arcsf')
Skipping update for 'buildsys': existing non-empty dblp_venue kept ('conf/sensys')
Skipping 'cca': acronym is duplicated in core_table
Skipping update for 'ccc': existing non-empty dblp_venue kept ('conf/coco')
Skipping update for 'cloud': existing non-empty dblp_venue kept ('conf/IEEEcloud')
Skipping update for 'ds': existing non-empty dblp_venue kept ('conf/dis')
Skipping update for 'ec': existing non-empty dblp_venue kept ('conf/sigecom')
Skipping update for 'evocop': existing non-empty dblp_venue kept ('conf/evoW;conf/evocop')
Skipping update for 'fg': existing non-empty dblp_venue kept ('conf/fgr')
Skipping 'gi': acronym is duplicated in core_table
Skipping update for 'i4cs': existing non-empty dblp_venue kept ('conf/iics;conf

In [2]:
json_path = Path('../dblp_venue_names.json')
if not json_path.exists():
    raise FileNotFoundError(f"{json_path} nem található.")
with open(json_path, 'r', encoding='utf-8') as fh:
    names = json.load(fh)

for crossref, name in names.items():
    rec = {"article": {
          "@key": "journals/cgf/abc24",
          "author": [],
          "title": "This should be classied as Rank A",
          "pages": "63-74",
          "year": "2020",
          "volume": "37",
          "journal": "Comput. Graph. Forum",
          "number": "2",
          "crossref": "journals/cgf/vc1",
        } }
    # rec= {"inproceedings": {
    #       "author": [
    #       ],
    #       "title": "This should be classied as Rank A*",
    #       "pages": "449-456",
    #       "year": "2003",
    #       "crossref": "conf/visualization/2003",
    #       "booktitle": "IEEE Visualization"
    #     }}
    key, record, rank, foreign_paper, short_paper, search_log, original_rank = classify_paper(rec)

NameError: name 'Path' is not defined

In [1]:
# Iterate over conference_stats/dblp_venue_names.json and classify a synthetic record for each dblp venue
import json
import os
import classify_paper

in_path = os.path.join('..','dblp_venue_names.json')
try:
    with open(in_path, 'r', encoding='utf-8') as f:
        venue_names = json.load(f)
except Exception as e:
    print(f"Hiba a {in_path} betöltésekor: {e}")
    venue_names = {}

def ascii_histogram(d, width=60, cap=12):
    def to_int(k):
        if str(k) == "null":
            return 0
        try:
            return int(k)
        except:
            return 0

    # aggregálás
    agg = {}
    for k, v in d.items():
        i = to_int(k)
        if i >= cap:
            agg[f"{cap}+"] = agg.get(f"{cap}+", 0) + v
        else:
            agg[i] = agg.get(i, 0) + v

    max_val = max(agg.values())

    for k in sorted(agg, key=lambda x: cap if x == f"{cap}+" else int(x)):
        v = agg[k]
        n = int(v / max_val * width)
        bar = "#" * n
        print(f"{str(k):>3} | {bar} ({v})")

results = {}
count = 0
for dblp_key, acr_map in venue_names.items():
    for acr,pubnum_map in acr_map:
        # rec = {"inproceedings": {
        #     "author": [],
        #     "title": f"Test classification for {dblp_key}",
        #     "pages": "1-10",
        #     "year": "2020",
        #     "crossref": f"{dblp_key}/2020",
        #     "booktitle": acronym
        # }}
        #key, record, rank, foreign_paper, short_paper, search_log, original_rank = classify_paper.classify_paper(rec)
        #print(f"{dblp_key} {acr} ->  {key} {rank} {'short_paper' if short_paper else ''} {original_rank}") # {search_log}
        last_rank, short_paper, acronym_row =classify_paper.core_rank(acr, f"{dblp_key}/2020", f"{dblp_key}/2020", 2026)
        if acronym_row is not None:
            if last_rank=='A*':
                if not short_paper:
                    print(f"{dblp_key} >{acr}< ->  {last_rank} {'short' if short_paper else 'full'} |{acronym_row['Acronym']}| {acronym_row['Name']}")
                    ascii_histogram(pubnum_map,20)
        else:
            print(f"{dblp_key} >{acr}< ->  No match")

    count += 1

print(f"Processed {count} venues.")


conf/acl >ACL< ->  A* full |ACL;ACL/IJCNLP| Association for Computational Linguistics
 -1 |  (1)
  0 | #### (938)
  1 |  (40)
  2 |  (101)
  3 |  (118)
  4 | # (288)
  5 | #### (819)
  6 | ## (515)
  7 | #### (901)
  8 | ## (413)
  9 | #### (974)
 10 | ##### (1050)
 11 | ## (478)
12+ | #################### (4034)
conf/acl >ACL/IJCNLP< ->  A* full |ACL;ACL/IJCNLP| Association for Computational Linguistics
  0 |  (2)
  3 | ###### (93)
  4 |  (4)
  5 | # (24)
  6 | ## (35)
  7 | ## (36)
  8 | ######### (136)
  9 | #### (72)
 10 | ######### (139)
 11 | ###### (90)
12+ | #################### (295)
conf/ismar >ISMAR< ->  A* full |ISMAR;ISAR;IWAR| IEEE/ACM International Symposium on Mixed and Augmented Reality
  0 | #### (108)
  1 | ################# (414)
  2 |  (1)
  3 | ##### (123)
  4 |  (6)
  5 | ## (63)
  6 | # (34)
  7 | ## (68)
  8 | ###### (146)
  9 | #################### (471)
 10 | # (30)
 11 | # (39)
12+ |  (15)
conf/ismar >IWAR< ->  A* full |ISMAR;ISAR;IWAR| IEEE/ACM Internationa